# 04 Tiny ViT：EncoderBlock 结构拆解

前面已经完成 Tiny ViT 输入部分的理论和 shape 路线：

```text
图片：B x 3 x 32 x 32
→ Patch Embedding
patch tokens：B x 64 x 192
→ 拼接 CLS token
→ 加入位置编码
Encoder 输入：B x 65 x 192
```

现在进入 Tiny ViT 的核心：**Transformer EncoderBlock**。

这一课只拆解结构、公式、shape 和参数量，不编写实现代码。等整个 Tiny ViT 结构真正理解清楚后，再在独立 Python 文件中亲手实现。

## 1. 本课学习目标

完成本课后，需要能够：

1. 独立画出一个 Pre-Norm EncoderBlock。
2. 说清 Multi-Head Self-Attention 和 FFN 的分工。
3. 从 `B x 65 x 192` 推导 Q、K、V 和注意力矩阵的 shape。
4. 解释 3 个 heads 怎样把 192 维拆成每个 head 64 维。
5. 解释缩放点积、Softmax 和加权求和分别做什么。
6. 解释多个 heads 怎样重新合并为 192 维。
7. 追踪 FFN 的 `192 → 768 → 192`。
8. 解释两次残差连接为什么都要求 shape 不变。
9. 区分 Pre-Norm 和 Post-Norm。
10. 手算一个 EncoderBlock 和 4 层 Encoder 的参数量。

## 2. EncoderBlock 接收到什么

当前 Tiny ViT 配置：

```text
patch 数量：64
加入 CLS 后的序列长度 N：65
模型维度 D：192
注意力头数 h：3
每个 head 的维度 d_head：64
FFN 隐藏维度 d_ff：768
Encoder 深度 depth：4
```

所以一个 EncoderBlock 的输入是：

$$
X:B\times65\times192
$$

第 0 个 token 是 CLS，后面 64 个 token 分别对应图片中的 64 个 patches。位置编码已经在进入 Encoder 前加好，EncoderBlock 内部不会反复添加位置编码。

## 3. EncoderBlock 的作用是什么

一个 EncoderBlock 可以先理解成两个连续阶段：

```text
第一阶段：Multi-Head Self-Attention
作用：让不同 tokens 之间交换信息

第二阶段：Feed Forward Network
作用：对每个 token 的内部特征进行非线性加工
```

它们的分工不能混淆：

- Attention 负责 token 与 token 之间的交流；
- FFN 不负责不同 token 之间交流，它对每个位置单独使用同一组 MLP 参数。

两部分外面还要配合 LayerNorm、残差连接和 Dropout，才能形成完整 EncoderBlock。

## 4. 本项目采用 Pre-Norm 结构

torchvision ViT 使用 Pre-Norm，也就是先归一化，再进入子层。完整结构是：

```text
输入 X
→ LayerNorm 1
→ Multi-Head Self-Attention
→ Dropout
→ 与原输入 X 做残差相加
→ 得到 H

H
→ LayerNorm 2
→ FFN
→ Dropout
→ 与 H 做残差相加
→ 得到输出 O
```

对应公式：

$$
H=X+Dropout(MHA(LN_1(X)))
$$

$$
O=H+Dropout(FFN(LN_2(H)))
$$

## 5. 先看完整 shape 主线

一个 EncoderBlock 的外部 shape 路线非常简单：

```text
输入：B x 65 x 192
→ LayerNorm：B x 65 x 192
→ MHA：B x 65 x 192
→ 第一次残差：B x 65 x 192
→ LayerNorm：B x 65 x 192
→ FFN：B x 65 x 768 → B x 65 x 192
→ 第二次残差：B x 65 x 192
→ 输出：B x 65 x 192
```

EncoderBlock 内部虽然会拆分 heads、产生 `65 x 65` 注意力矩阵并扩张 FFN 维度，但最终必须回到 `B x 65 x 192`。这样才能进行残差相加，也才能连续堆叠多个 Blocks。

## 6. 第一步：LayerNorm 1

输入：

$$
X:B\times65\times192
$$

LayerNorm 对每个 token 的 192 个特征进行归一化。它不会混合不同 tokens，也不会改变 shape：

$$
LN_1(X):B\times65\times192
$$

可以理解成 batch 中每张图片的每个 token 都单独执行：

$$
x_{token}\in\mathbb{R}^{192}
\longrightarrow
LN(x_{token})\in\mathbb{R}^{192}
$$

LayerNorm 自己有可学习的缩放参数和偏移参数，各 192 个。

## 7. 第二步：生成 Q、K、V

归一化后的每个 token 要分别产生 Query、Key、Value：

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V
$$

理论上可以使用三个 Linear。工程中通常使用一个 `D → 3D` 的大 Linear 一次得到 QKV，再沿最后一维切成三份。

当前 $D=192$：

```text
输入：B x 65 x 192
QKV 投影：B x 65 x 576
切分后：
Q：B x 65 x 192
K：B x 65 x 192
V：B x 65 x 192
```

Q、K、V 来自同一份输入，所以这里是 Self-Attention；但它们使用不同的可学习投影，因此承担不同职责。

## 8. 第三步：把 192 维拆成 3 个 heads

注意力头数：

$$
h=3
$$

每个 head 的维度：

$$
d_{head}=D/h=192/3=64
$$

因此 Q、K、V 都要经历：

$$
B\times65\times192
\longrightarrow
B\times65\times3\times64
\longrightarrow
B\times3\times65\times64
$$

最终把 head 维提前，是为了让 3 个 heads 并行执行注意力计算。

`embed_dim` 必须能被 `num_heads` 整除，否则不能均匀拆分。例如 192 可以分成 3 x 64 或 6 x 32，但不能平均分成 5 个 heads。

## 9. 第四步：计算注意力分数

每个 head 内部计算：

$$
Scores=QK^T
$$

Q 的 shape：

$$
B\times3\times65\times64
$$

K 转置最后两个维度后：

$$
B\times3\times64\times65
$$

矩阵乘法得到：

$$
Scores:B\times3\times65\times65
$$

每个 head 都会产生一张 65 x 65 的关系表。每一行表示一个 Query token 对全部 65 个 Key tokens 的匹配分数。

### 65 x 65 注意力矩阵里有什么

序列包含 1 个 CLS 和 64 个 patch tokens，所以矩阵同时包含：

```text
CLS 查询 CLS
CLS 查询各个 patches
每个 patch 查询 CLS
每个 patch 查询其他所有 patches
```

第 0 行特别重要：它描述 CLS 在当前 head 中准备从自己和 64 个 patches 汇总多少信息。

Encoder 使用双向 Self-Attention，没有 GPT 中的 causal mask。当前图片的每个 token 都可以查看完整序列。CIFAR 图片长度固定，也暂时不需要 padding mask。

## 10. 第五步：为什么要除以根号 d_head

缩放点积注意力使用：

$$
Scores=\frac{QK^T}{\sqrt{d_{head}}}
$$

当前：

$$
\sqrt{d_{head}}=\sqrt{64}=8
$$

维度较大时，点积数值容易变大。过大的分数进入 Softmax 后可能使概率过于接近 0 或 1，导致梯度变小、训练不稳定。

除以根号 $d_{head}$ 可以控制分数尺度。shape 不会变化，仍然是：

$$
B\times3\times65\times65
$$

## 11. 第六步：Softmax 得到注意力权重

对分数矩阵最后一个维度，也就是 Key token 维度执行 Softmax：

$$
A=Softmax(Scores,\ dim=-1)
$$

输出 shape 不变：

$$
A:B\times3\times65\times65
$$

每一行的 65 个权重满足：

$$
A_{i,j}\ge0,\qquad\sum_{j=1}^{65}A_{i,j}=1
$$

含义是：对于某个 Query token，模型把总计 100% 的关注程度分配给全部 65 个 Value 来源。

## 12. 第七步：用注意力权重汇总 V

注意力输出：

$$
Z=AV
$$

shape 计算：

$$
(B\times3\times65\times65)
\times
(B\times3\times65\times64)
\longrightarrow
B\times3\times65\times64
$$

每个 Query token 都会按照自己的 65 个注意力权重，对 65 个 Value 向量做加权求和。

输出仍保留 65 个 token，但每个 token 已经融合了其他位置的信息。

## 13. 第八步：合并多个 heads

3 个 heads 的输出是：

$$
B\times3\times65\times64
$$

先把 token 维移回前面，再把 3 和 64 合并：

$$
B\times3\times65\times64
\longrightarrow
B\times65\times3\times64
\longrightarrow
B\times65\times192
$$

随后还要经过输出投影 $W_O$：

$$
B\times65\times192
\longrightarrow
B\times65\times192
$$

输出投影让不同 heads 提取的关系信息再次混合，而不是简单拼接后直接结束。

## 14. MHA 完整 shape 总结

```text
输入 X                     B x 65 x 192
LayerNorm                  B x 65 x 192
QKV 总投影                 B x 65 x 576
Q / K / V                  B x 65 x 192
拆分 heads                 B x 3 x 65 x 64
QK^T                       B x 3 x 65 x 65
缩放 + Softmax             B x 3 x 65 x 65
Attention weights x V      B x 3 x 65 x 64
合并 heads                 B x 65 x 192
输出投影                   B x 65 x 192
```

MHA 内部 shape 变化很多，但对 EncoderBlock 外部来说，它完成的是：

$$
B\times65\times192
\longrightarrow
B\times65\times192
$$

## 15. 第一次残差连接

MHA 输出后进行：

$$
H=X+Dropout(MHA(LN_1(X)))
$$

两个被相加的张量都是：

$$
B\times65\times192
$$

残差连接可以理解成：

```text
保留原来的 token 表示 X
+
加入 Attention 学到的信息增量
```

如果 Attention 当前还没有学到有用更新，残差路径仍然允许原信息继续向后传递，也为深层网络中的梯度传播提供更直接的路径。

## 16. 第二部分：Position-wise FFN

第一次残差得到：

$$
H:B\times65\times192
$$

先经过第二个 LayerNorm，再进入 FFN：

```text
Linear 1：192 → 768
GELU
Dropout
Linear 2：768 → 192
Dropout
```

shape 路线：

$$
B\times65\times192
\longrightarrow
B\times65\times768
\longrightarrow
B\times65\times192
$$

FFN 通常把特征维扩张到 4 倍，再压回原来的 $D$。中间的非线性让模型可以学习比单纯线性变换更复杂的特征组合。

### 为什么叫 Position-wise FFN

FFN 对 65 个 token 分别执行相同的 MLP：

```text
CLS token      → 同一个 FFN
patch token 1  → 同一个 FFN
patch token 2  → 同一个 FFN
...
patch token 64 → 同一个 FFN
```

它不会把 token 1 和 token 2 直接放进同一次 Linear 中混合。不同 tokens 之间的信息交流已经由前面的 Self-Attention 完成。

可以把二者分工记成：

```text
Attention：横向交流——不同位置互相看
FFN：纵向加工——每个位置单独深化特征
```

## 17. 第二次残差连接

FFN 输出后进行：

$$
O=H+Dropout(FFN(LN_2(H)))
$$

FFN 虽然中间扩张到 768 维，但最后必须回到 192 维，才能与 $H$ 相加。

因此：

$$
H:B\times65\times192
$$

$$
FFN(LN_2(H)):B\times65\times192
$$

$$
O:B\times65\times192
$$

至此，一个完整 EncoderBlock 结束。

## 18. Pre-Norm 与 Post-Norm 的区别

### Pre-Norm

$$
X+Sublayer(LN(X))
$$

先 LayerNorm，再进入 Attention 或 FFN。torchvision ViT 使用这种结构。

### Post-Norm

$$
LN(X+Sublayer(X))
$$

先执行子层和残差相加，最后再 LayerNorm。原始 Transformer 论文经典图示常使用这种结构。

两者包含的零件相同，但顺序不同。以后自己实现 Tiny ViT 时要选择一种并保持公式、forward 和结构图一致，不能在解释时说 Pre-Norm，代码却写成 Post-Norm。

## 19. CLS 在 EncoderBlock 中怎样变化

进入第一个 Block 前，CLS 只是共享的可学习向量加上位置编码。

在 Self-Attention 中，CLS 作为第 0 个 Query，会对全部 65 个 Keys 计算权重，并汇总全部 Values：

```text
CLS_new
= CLS 自身信息
+ patch 1 信息
+ patch 2 信息
+ ...
+ patch 64 信息
```

之后 CLS 还会经过自己的 FFN 非线性加工。

每经过一个 Block，CLS 都会重新与更新后的 patch tokens 交流。因此堆叠多层后，CLS 逐渐形成适合分类的整图表示。

## 20. 堆叠 4 个 EncoderBlocks

Tiny ViT 建议先使用 depth=4：

```text
输入：B x 65 x 192
→ EncoderBlock 1：B x 65 x 192
→ EncoderBlock 2：B x 65 x 192
→ EncoderBlock 3：B x 65 x 192
→ EncoderBlock 4：B x 65 x 192
→ Encoder 输出：B x 65 x 192
```

四层的输入输出 shape 相同，但 token 内容不是原地不变。每个 Block 都会重新计算注意力关系，再进行一次 FFN 加工。

不同 Blocks 不共享参数。Block 1 和 Block 2 都有自己的 QKV、输出投影、FFN 和 LayerNorm 参数。

## 21. 手算 MHA 参数量

### QKV 投影

一个 `D → 3D` 的 Linear：

$$
3D^2+3D
$$

代入 $D=192$：

$$
3\times192\times192+3\times192=111168
$$

### 输出投影

一个 `D → D` 的 Linear：

$$
D^2+D=192\times192+192=37056
$$

### MHA 总参数量

$$
111168+37056=148224
$$

注意力分数矩阵没有可学习参数，它是根据当前输入动态计算出来的。

## 22. 手算 FFN 和 LayerNorm 参数量

### Linear 1：192 → 768

$$
192\times768+768=148224
$$

### Linear 2：768 → 192

$$
768\times192+192=147648
$$

### FFN 总参数量

$$
148224+147648=295872
$$

### 两个 LayerNorm

每个 LayerNorm 有 192 个缩放参数和 192 个偏移参数：

$$
2\times(192+192)=768
$$

Dropout、残差相加、Softmax 和 GELU 都没有可学习参数。

## 23. 一个 Block 和 4 层 Encoder 的参数量

一个 EncoderBlock：

$$
MHA+FFN+LayerNorms
$$

$$
148224+295872+768=444864
$$

4 个不共享参数的 Blocks：

$$
4\times444864=1779456
$$

可以看到，即使图片很小，Encoder 仍然是 Tiny ViT 参数量的主体。FFN 参数量约为 MHA 的两倍，因此不能只关注 Attention 而忽略 FFN。

## 24. 注意力计算量为什么与 token 数量平方相关

每个 head 都要产生 $N\times N$ 分数表。当前加入 CLS 后 $N=65$：

$$
65\times65=4225
$$

3 个 heads、batch_size 为 $B$ 时，分数元素数量为：

$$
B\times3\times65\times65
$$

如果 patch_size 变小，patch 数量会迅速增加，注意力矩阵按平方增长。

这解释了为什么前面比较 patch_size=4 和 8 时，token 数量只减少到 1/4，但 $N^2$ 关系数量大约减少到 1/16。

## 25. 与 torchvision EncoderBlock 对应

上一课预训练 ViT 源码中看到：

```text
ln_1             → 第一处 LayerNorm
self_attention   → Multi-Head Self-Attention
dropout          → Attention 输出后的 Dropout
x + input        → 第一次残差连接
ln_2             → 第二处 LayerNorm
mlp              → FFN：Linear + GELU + Linear
x + y            → 第二次残差连接
```

我们之后自己写 Tiny ViT 时，类名和变量名可以不同，但这些结构职责和 shape 约束必须一致。

## 26. 常见误解

### 误解一：3 个 heads 会把输出维度变成 576

不会。总维度 192 被拆成 3 个 64 维 heads，合并后仍是 192。

### 误解二：QKV 投影输出 576，所以 Encoder 输出也是 576

不是。576 只是把 Q、K、V 暂时放在一个张量中，切分后每个仍是 192 维。

### 误解三：注意力矩阵的 65 x 65 是模型参数

不是。它根据每个输入动态计算，不保存在模型参数中。

### 误解四：FFN 会让不同 patches 交流

不会。FFN 对每个 token 单独处理，不跨 token 维混合。

### 误解五：FFN 扩张到 768 后 Block 输出也是 768

不是。第二个 Linear 会压回 192，保证残差连接和多层堆叠。

### 误解六：位置编码在每个 Block 中重新加入

本项目中只在进入 Encoder 前加入一次。之后位置相关信息随着 token 表示一起传播。

## 27. 纸上推导练习

在开始写 Python 实现前，建议先完成下面练习。

### 练习一：独立画出 EncoderBlock

要求标出两个 LayerNorm、MHA、FFN、两个 Dropout 和两条残差路径。

### 练习二：写出 MHA shape

从 `B x 65 x 192` 开始，不看笔记写到注意力输出重新回到 `B x 65 x 192`。

### 练习三：修改 heads

如果 heads 从 3 改成 6，写出 $d_{head}$、Q/K/V 分头后的 shape 和注意力矩阵 shape。

### 练习四：修改 patch_size

如果 patch_size=8，序列长度变成 17。重新写出 Q、K、V、Scores 和 MHA 输出 shape。

### 练习五：重新手算参数量

保持 $D=192$，把 $d_{ff}$ 从 768 改成 384，重新计算一个 Block 的 FFN 参数量和总参数量。

## 27.1 纸上推导练习参考答案

### 练习一答案：EncoderBlock 结构图

这里使用的是 ViT 中常见的 Pre-LN 结构，也就是先 LayerNorm，再进入子模块。

```text
输入 X: B x 65 x 192

第一段：注意力子层
X
├─────────────── 残差路径 1 ───────────────┐
│                                           ↓
└→ LayerNorm 1 → MHA → Dropout → 加回 X → X1

第二段：FFN 子层
X1
├─────────────── 残差路径 2 ────────────────┐
│                                           ↓
└→ LayerNorm 2 → FFN → Dropout → 加回 X1 → 输出 X2
```

两个残差相加都要求主路径输出和残差路径输入形状一致，所以 MHA 输出必须是 `B x 65 x 192`，FFN 最后也必须压回 `B x 65 x 192`。

### 练习二答案：heads=3 时的 MHA shape

已知：

$$
D=192,\qquad heads=3,\qquad d_{head}=192/3=64
$$

完整 shape 流程：

```text
输入 X
B x 65 x 192

一次线性层生成 QKV
B x 65 x 576

切成 Q、K、V
Q: B x 65 x 192
K: B x 65 x 192
V: B x 65 x 192

拆成 3 个 heads
Q: B x 3 x 65 x 64
K: B x 3 x 65 x 64
V: B x 3 x 65 x 64

计算注意力分数 QK^T
Scores: B x 3 x 65 x 65

Softmax 后得到注意力权重
Attention Weights: B x 3 x 65 x 65

注意力权重乘以 V
Context: B x 3 x 65 x 64

合并 heads
B x 65 x 192

输出线性层
B x 65 x 192
```

所以注意力内部虽然临时拆成多个 head，但最终仍然回到 `B x 65 x 192`。

### 练习三答案：heads 从 3 改成 6

保持总维度 $D=192$ 不变：

$$
d_{head}=192/6=32
$$

此时每个 head 变窄，但 head 数量更多。

```text
输入 X
B x 65 x 192

Q、K、V 分头后
Q: B x 6 x 65 x 32
K: B x 6 x 65 x 32
V: B x 6 x 65 x 32

注意力矩阵
Scores: B x 6 x 65 x 65

注意力输出
Context: B x 6 x 65 x 32

合并 heads 后
B x 65 x 192

输出线性层后
B x 65 x 192
```

注意：heads 变多不代表输出维度变成 `B x 65 x 1152`。总维度仍然是 192，只是被拆成了 6 份。

### 练习四答案：patch_size=8，序列长度变成 17

CIFAR-10 图片大小是 `32 x 32`。当 `patch_size=8` 时：

$$
patch\ 数量=(32/8)\times(32/8)=4\times4=16
$$

加入 1 个 CLS token 后：

$$
N=16+1=17
$$

如果仍然使用 `D=192, heads=3`，则：

$$
d_{head}=192/3=64
$$

shape 流程变成：

```text
输入 X
B x 17 x 192

Q、K、V 分头后
Q: B x 3 x 17 x 64
K: B x 3 x 17 x 64
V: B x 3 x 17 x 64

注意力分数
Scores: B x 3 x 17 x 17

注意力权重乘以 V
Context: B x 3 x 17 x 64

合并 heads
B x 17 x 192

MHA 输出
B x 17 x 192
```

和 `patch_size=4` 相比，主要变化是 token 数从 65 降到 17，注意力矩阵从 `65 x 65` 降到 `17 x 17`。

### 练习五答案：d_ff 从 768 改成 384

保持 $D=192$，只把 FFN 中间维度改成 $d_{ff}=384$。

第一层 Linear：`192 -> 384`

$$
192\times384+384=73728+384=74112
$$

第二层 Linear：`384 -> 192`

$$
384\times192+192=73728+192=73920
$$

所以 FFN 参数量为：

$$
74112+73920=148032
$$

MHA 参数量保持不变：

$$
148224
$$

两个 LayerNorm 参数量保持不变：

$$
2\times(192+192)=768
$$

一个 EncoderBlock 的新参数量为：

$$
148224+148032+768=297024
$$

如果堆叠 4 个不共享参数的 EncoderBlock：

$$
4\times297024=1188096
$$

这个结果说明：把 $d_{ff}$ 减半，会明显降低 FFN 和整个 EncoderBlock 的参数量，但也可能降低每个 token 的非线性表达能力。

## 28. 后续自己写 Python 文件时的实现顺序

理论确认后，不要一次写完整 Tiny ViT。建议按下面顺序逐个验证：

```text
1. MultiHeadSelfAttention
   先只检查输出和 attention weights shape

2. FeedForward
   检查 192 → 768 → 192

3. EncoderBlock
   加入 LayerNorm 和两次残差连接

4. Encoder
   堆叠 4 个 Blocks

5. 与前面的 ViTInputEmbedding 连接

6. 取 CLS 并加入分类头

7. 假输入 forward、假 loss、backward
```

每完成一个模块就验证 shape，不要等整个模型写完才第一次运行。

## 29. 本节小结

一个 Tiny ViT EncoderBlock 的核心是：

```text
X
→ LN → MHA → Dropout → 残差相加
→ LN → FFN → Dropout → 残差相加
→ O
```

完整 shape 主线：

```text
B x 65 x 192
→ Q/K/V：B x 3 x 65 x 64
→ Scores：B x 3 x 65 x 65
→ MHA 输出：B x 65 x 192
→ 第一次残差：B x 65 x 192
→ FFN：B x 65 x 768 → B x 65 x 192
→ 第二次残差：B x 65 x 192
```

一个 Block 有 444864 个参数，4 个 Blocks 有 1779456 个参数。

下一课将继续从结构层面组装完整 Tiny ViT：输入模块、4 层 Encoder、最终 LayerNorm、CLS 提取和分类头。仍然先写 Notebook，不提前替代后续 Python 实现。

### 完成本课后的掌握标准

- 能独立画出 Pre-Norm EncoderBlock；
- 能写出 QKV 投影和拆分 heads 的全部 shape；
- 能解释 `B x 3 x 65 x 65` 每个维度；
- 能解释除以根号 64 的原因；
- 能解释 Softmax 为什么沿最后一维；
- 能把多个 heads 合并回 192 维；
- 能解释 MHA 和 FFN 的分工；
- 能说明两次残差连接的 shape 要求；
- 能区分 Pre-Norm 和 Post-Norm；
- 能手算一个 Block 和 4 层 Encoder 的参数量。

## 30. 自测问题

1. 一个 EncoderBlock 的两个主要子层是什么？
2. Pre-Norm 中 LayerNorm 位于子层之前还是之后？
3. 为什么 QKV 总投影输出是 576 维？
4. 192 维使用 3 个 heads 时，每个 head 是多少维？
5. Q 和 K 相乘后为什么得到 `B x 3 x 65 x 65`？
6. 为什么注意力分数要除以 8？
7. Softmax 为什么沿最后一个 65 维计算？
8. 注意力权重乘 V 后为什么回到 `B x 3 x 65 x 64`？
9. 合并 heads 后为什么是 192 维而不是 576 维？
10. 输出投影 $W_O$ 有什么作用？
11. MHA 与 FFN 的分工有什么不同？
12. FFN 为什么要从 192 扩张到 768，再压回 192？
13. 两次残差连接为什么都要求子层输出回到 192 维？
14. 4 个 Blocks 是否共享 QKV 和 FFN 参数？
15. 一个 Block 和 4 层 Encoder 分别有多少参数？

### 自测参考答案

1. Multi-Head Self-Attention 和 Position-wise FFN。
2. 位于子层之前。
3. 因为一次同时产生 Q、K、V，即 $3D=3\times192=576$。
4. $192/3=64$ 维。
5. 每个 head 的 65 个 Queries 都要与 65 个 Keys 计算点积。
6. $\sqrt{d_{head}}=\sqrt{64}=8$，缩放可避免分数过大导致 Softmax 饱和。
7. 每个 Query 要在 65 个 Key 来源之间分配总和为 1 的注意力权重。
8. 65 个权重对 65 个 64 维 Value 加权求和，每个 Query 得到一个 64 维结果。
9. 192 是总模型维度，被拆成 3 个 64 维 heads；合并后是 $3\times64=192$。
10. 让不同 heads 的输出信息重新混合，并保持最终维度为 $D$。
11. MHA 负责 tokens 之间交流；FFN 负责每个 token 内部的非线性加工。
12. 扩张提供更大的非线性表达空间，压回 192 用于残差连接和多层堆叠。
13. 残差相加要求两个张量 shape 完全一致。
14. 不共享，每个 Block 有自己独立的一套参数。
15. 一个 Block 为 444864，4 层为 1779456。